RetailPulse 360

Notebook 08 — Demand Forecasting

Goal: Predict future demand at store-product level, directly informed by everything EDA
(Notebook 07) told us — real multi-seasonality (weekly + Eid + wedding season), store-size
as a meaningful predictor, and the need for explicit calendar features rather than a naive
single-seasonality approach. This model is also the direct input to Phase 4's redistribution
engine — "how much will this store need" is what makes "should we move stock here" possible.

Input: sales.csv, stores.csv, skus.csv, products.csv, store_personalities.csv
Output: demand_forecast_model (saved), forecast_results.csv

In [2]:
# 1. IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
# 2. LOAD DATA AND AGGREGATE TO STORE+PRODUCT+DATE
# ============================================================

BASE_PATH = "/kaggle/input/datasets/hamaz911/notebook-8-dataset/"

sales = pd.read_csv(BASE_PATH + "sales.csv", parse_dates=["date"])
skus = pd.read_csv(BASE_PATH + "skus.csv")
stores = pd.read_csv(BASE_PATH + "stores.csv")
products = pd.read_csv(BASE_PATH + "products.csv")
store_personalities = pd.read_csv(BASE_PATH + "store_personalities.csv")

# Aggregate raw sales up to store+product+date (matches Notebook 06/07's granularity)
sales_sku = sales.merge(skus[["sku_id", "product_id"]], on="sku_id", how="left")
daily_demand = sales_sku.groupby(["store_id", "product_id", "date"])["units_sold"].sum().reset_index()

print("Aggregated rows:", len(daily_demand))
print("Unique store-product combos:", daily_demand.groupby(["store_id","product_id"]).ngroups)
print("Date range:", daily_demand["date"].min(), "to", daily_demand["date"].max())

Aggregated rows: 2108309
Unique store-product combos: 13669
Date range: 2024-02-23 00:00:00 to 2026-08-23 00:00:00


In [4]:
# 3. DENSIFY THE DEMAND TABLE (INCLUDE TRUE ZERO-DEMAND DAYS)
# ============================================================
# Forecasting requires learning the true zero-inflated distribution of
# demand, not just "how many units given a sale happened." Sparse data
# alone would make the model systematically overestimate demand.

all_combos = daily_demand[["store_id", "product_id"]].drop_duplicates()
all_dates = pd.DataFrame({"date": pd.date_range(daily_demand["date"].min(), daily_demand["date"].max(), freq="D")})

all_combos["key"] = 1
all_dates["key"] = 1
full_grid = all_combos.merge(all_dates, on="key").drop(columns="key")

print("Full dense grid size:", len(full_grid))

dense_demand = full_grid.merge(daily_demand, on=["store_id", "product_id", "date"], how="left")
dense_demand["units_sold"] = dense_demand["units_sold"].fillna(0)

print("Dense demand table:", dense_demand.shape)
print("Zero-demand days:", (dense_demand["units_sold"] == 0).sum(),
      f"({(dense_demand['units_sold']==0).mean()*100:.1f}%)")
print("\nData quality check:")
print("Missing values:", dense_demand.isna().sum().sum())
print("Negative units_sold:", (dense_demand["units_sold"] < 0).sum())

Full dense grid size: 12479797
Dense demand table: (12479797, 4)
Zero-demand days: 10371488 (83.1%)

Data quality check:
Missing values: 0
Negative units_sold: 0


In [5]:
# 4. BUILD CALENDAR FEATURES
# ============================================================
# EDA (Notebook 07) confirmed we need EXPLICIT calendar features since
# a single automatic decomposition can't separate overlapping weekly +
# Eid + wedding-season cycles. Same real, sourced Eid dates as before.

EID_DATES = [
    "2024-04-10", "2024-06-17", "2025-03-31",
    "2025-06-07", "2026-03-20", "2026-05-27",
]
PRE_EID_WINDOW = 10

dense_demand["day_of_week"] = dense_demand["date"].dt.dayofweek  # Monday=0
dense_demand["month"] = dense_demand["date"].dt.month
dense_demand["is_wedding_season"] = dense_demand["month"].isin([11, 12, 1, 2]).astype(int)

dense_demand["is_pre_eid"] = 0
for eid in EID_DATES:
    eid_ts = pd.Timestamp(eid)
    mask = (dense_demand["date"] >= eid_ts - pd.Timedelta(days=PRE_EID_WINDOW)) & (dense_demand["date"] < eid_ts)
    dense_demand.loc[mask, "is_pre_eid"] = 1

print("Calendar features built:")
print(dense_demand[["day_of_week", "month", "is_wedding_season", "is_pre_eid"]].describe())
print("\nPre-Eid days flagged:", dense_demand["is_pre_eid"].sum())

Calendar features built:
        day_of_week         month  is_wedding_season    is_pre_eid
count  1.247980e+07  1.247980e+07       1.247980e+07  1.247980e+07
mean   3.006572e+00  6.272727e+00       2.705367e-01  6.571742e-02
std    2.000537e+00  3.221827e+00       4.442371e-01  2.477875e-01
min    0.000000e+00  1.000000e+00       0.000000e+00  0.000000e+00
25%    1.000000e+00  4.000000e+00       0.000000e+00  0.000000e+00
50%    3.000000e+00  6.000000e+00       0.000000e+00  0.000000e+00
75%    5.000000e+00  9.000000e+00       1.000000e+00  0.000000e+00
max    6.000000e+00  1.200000e+01       1.000000e+00  1.000000e+00

Pre-Eid days flagged: 820140


In [6]:
# 5. MERGE STORE + PRODUCT FEATURES
# ============================================================
# store_size, region, and Rossmann personality (dow ratios, promo_lift,
# trend) already live in stores.csv from Notebook 02. gender/category
# come from products.csv (Notebook 05).

store_features = stores[["store_id", "store_size", "region",
                          "promo_lift", "trend_pct_per_year", "volatility_cv"]]
dense_demand = dense_demand.merge(store_features, on="store_id", how="left")

product_features = products[["product_id", "gender", "category", "price_pkr"]]
dense_demand = dense_demand.merge(product_features, on="product_id", how="left")

print("Shape after merging features:", dense_demand.shape)
print("Missing values after merge:", dense_demand.isna().sum()[dense_demand.isna().sum() > 0])

Shape after merging features: (12479797, 16)
Missing values after merge: Series([], dtype: int64)


In [7]:
# 6. LAG AND ROLLING-WINDOW FEATURES
# ============================================================
# Sort first — lag/rolling features only make sense in chronological
# order, per store-product group.

dense_demand = dense_demand.sort_values(["store_id", "product_id", "date"])

grp = dense_demand.groupby(["store_id", "product_id"])["units_sold"]

dense_demand["lag_7"] = grp.shift(7)     # same weekday, last week
dense_demand["lag_14"] = grp.shift(14)   # same weekday, two weeks ago
dense_demand["rolling_mean_7"] = grp.transform(lambda x: x.shift(1).rolling(7).mean())
dense_demand["rolling_mean_28"] = grp.transform(lambda x: x.shift(1).rolling(28).mean())

print("New feature columns added. Sample:")
print(dense_demand[["store_id","product_id","date","units_sold","lag_7","lag_14",
                     "rolling_mean_7","rolling_mean_28"]].head(10))

print("\nMissing values in new features (expected at the very start of each series):")
print(dense_demand[["lag_7","lag_14","rolling_mean_7","rolling_mean_28"]].isna().sum())

New feature columns added. Sample:
   store_id product_id       date  units_sold  lag_7  lag_14  rolling_mean_7  \
0  STY-0001  PROD-0001 2024-02-23         0.0    NaN     NaN             NaN   
1  STY-0001  PROD-0001 2024-02-24         0.0    NaN     NaN             NaN   
2  STY-0001  PROD-0001 2024-02-25         0.0    NaN     NaN             NaN   
3  STY-0001  PROD-0001 2024-02-26         0.0    NaN     NaN             NaN   
4  STY-0001  PROD-0001 2024-02-27         0.0    NaN     NaN             NaN   
5  STY-0001  PROD-0001 2024-02-28         0.0    NaN     NaN             NaN   
6  STY-0001  PROD-0001 2024-02-29         0.0    NaN     NaN             NaN   
7  STY-0001  PROD-0001 2024-03-01         0.0    0.0     NaN             0.0   
8  STY-0001  PROD-0001 2024-03-02         0.0    0.0     NaN             0.0   
9  STY-0001  PROD-0001 2024-03-03         2.0    0.0     NaN             0.0   

   rolling_mean_28  
0              NaN  
1              NaN  
2              NaN  


In [8]:
# 7. DROP INCOMPLETE ROWS, THEN TIME-BASED TRAIN/TEST SPLIT
# ============================================================
# Drop rows missing rolling_mean_28 (the largest window) — this
# automatically covers rows missing the shorter lag/rolling features too,
# since those windows are smaller and would already be filled by then.

model_data = dense_demand.dropna(subset=["rolling_mean_28"]).copy()
print("Rows after dropping incomplete history:", len(model_data))

# Categorical features -> pandas 'category' dtype (LightGBM handles
# these natively, no manual one-hot encoding needed)
CATEGORICAL_COLS = ["store_size", "region", "gender", "category", "day_of_week"]
for col in CATEGORICAL_COLS:
    model_data[col] = model_data[col].astype("category")

# Time-based split: hold out the LAST 90 days as test data.
# This window happens to include the most recent real Eid (May 27, 2026)
# and its pre-Eid window -- a genuinely meaningful test of whether the
# model captures seasonal spikes, not just an arbitrary cutoff.
TEST_DAYS = 90
cutoff_date = model_data["date"].max() - pd.Timedelta(days=TEST_DAYS)

train = model_data[model_data["date"] <= cutoff_date]
test = model_data[model_data["date"] > cutoff_date]

print(f"\nCutoff date: {cutoff_date.date()}")
print(f"Train: {len(train):,} rows, {train['date'].min().date()} to {train['date'].max().date()}")
print(f"Test:  {len(test):,} rows, {test['date'].min().date()} to {test['date'].max().date()}")
print(f"\nDoes test period include the last real Eid (2026-05-27)? "
      f"{(test['date'] <= pd.Timestamp('2026-05-27')).any() and (test['date'] >= pd.Timestamp('2026-05-17')).any()}")

Rows after dropping incomplete history: 12097065

Cutoff date: 2026-05-25
Train: 10,866,855 rows, 2024-03-22 to 2026-05-25
Test:  1,230,210 rows, 2026-05-26 to 2026-08-23

Does test period include the last real Eid (2026-05-27)? True


In [9]:
# 8. NAIVE BASELINE — "SAME AS LAST WEEK"
# ============================================================

test_eval = test.dropna(subset=["lag_7"]).copy()  # a few rows may lack lag_7 near the cutoff
naive_mae = mean_absolute_error(test_eval["units_sold"], test_eval["lag_7"])
naive_rmse = np.sqrt(mean_squared_error(test_eval["units_sold"], test_eval["lag_7"]))

print(f"Naive baseline (lag_7) — MAE: {naive_mae:.4f}, RMSE: {naive_rmse:.4f}")
print(f"Evaluated on {len(test_eval):,} test rows")

Naive baseline (lag_7) — MAE: 0.2940, RMSE: 0.6211
Evaluated on 1,230,210 test rows


In [10]:
# 9. TRAIN LIGHTGBM (POISSON OBJECTIVE FOR COUNT DATA)
# ============================================================

FEATURE_COLS = [
    "day_of_week", "month", "is_wedding_season", "is_pre_eid",
    "store_size", "region", "promo_lift", "trend_pct_per_year", "volatility_cv",
    "gender", "category", "price_pkr",
    "lag_7", "lag_14", "rolling_mean_7", "rolling_mean_28",
]

# Carve out the last 30 days of TRAIN (not test) as a validation set,
# purely for early stopping -- test stays fully untouched until final evaluation.
val_cutoff = train["date"].max() - pd.Timedelta(days=30)
train_fit = train[train["date"] <= val_cutoff]
train_val = train[train["date"] > val_cutoff]

X_train, y_train = train_fit[FEATURE_COLS], train_fit["units_sold"]
X_val, y_val = train_val[FEATURE_COLS], train_val["units_sold"]
X_test, y_test = test_eval[FEATURE_COLS], test_eval["units_sold"]

model = lgb.LGBMRegressor(
    objective="poisson",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
)

pred = model.predict(X_test)
pred = np.clip(pred, 0, None)  # demand can't be negative

model_mae = mean_absolute_error(y_test, pred)
model_rmse = np.sqrt(mean_squared_error(y_test, pred))

print(f"\nLightGBM — MAE: {model_mae:.4f}, RMSE: {model_rmse:.4f}")
print(f"Naive baseline — MAE: {naive_mae:.4f}, RMSE: {naive_rmse:.4f}")
print(f"MAE improvement over naive: {(1 - model_mae/naive_mae)*100:.1f}%")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.755839 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 749
[LightGBM] [Info] Number of data points in the train set: 10456785, number of used features: 16
[LightGBM] [Info] Start training from score -1.599825
Training until validation scores don't improve for 30 rounds
[50]	valid_0's poisson: 0.482913
[100]	valid_0's poisson: 0.475681
[150]	valid_0's poisson: 0.474232
[200]	valid_0's poisson: 0.473738
[250]	valid_0's poisson: 0.473516
[300]	valid_0's poisson: 0.473388
Did not meet early stopping. Best iteration is:
[300]	valid_0's poisson: 0.473388

LightGBM — MAE: 0.2747, RMSE: 0.4371
Naive baseline — MAE: 0.2940, RMSE: 0.6211
MAE improvement over naive: 6.5%


In [11]:
# 10. VERIFY — IS THE MODEL'S EDGE CONCENTRATED ON SPIKE DAYS?
# ============================================================

test_eval = test_eval.copy()
test_eval["prediction"] = pred

for segment_name, mask in [
    ("Pre-Eid days", test_eval["is_pre_eid"] == 1),
    ("Normal days", test_eval["is_pre_eid"] == 0),
]:
    seg = test_eval[mask]
    naive_seg_mae = mean_absolute_error(seg["units_sold"], seg["lag_7"])
    model_seg_mae = mean_absolute_error(seg["units_sold"], seg["prediction"])
    print(f"{segment_name} (n={len(seg):,}):")
    print(f"  Naive MAE: {naive_seg_mae:.4f} | Model MAE: {model_seg_mae:.4f} | "
          f"Improvement: {(1-model_seg_mae/naive_seg_mae)*100:.1f}%")

Pre-Eid days (n=13,669):
  Naive MAE: 0.4082 | Model MAE: 0.3666 | Improvement: 10.2%
Normal days (n=1,216,541):
  Naive MAE: 0.2927 | Model MAE: 0.2737 | Improvement: 6.5%


In [12]:
# 11. FEATURE IMPORTANCE
# ============================================================

importance = pd.DataFrame({
    "feature": FEATURE_COLS,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

print("Feature importance (which signals actually drive predictions):")
print(importance.to_string(index=False))

Feature importance (which signals actually drive predictions):
           feature  importance
       day_of_week        1280
   rolling_mean_28        1111
trend_pct_per_year         992
        promo_lift         912
            gender         828
        is_pre_eid         668
     volatility_cv         651
        store_size         629
 is_wedding_season         588
          category         528
         price_pkr         365
             month         275
            region         157
    rolling_mean_7          10
            lag_14           4
             lag_7           2


In [13]:
# 12. SAVE MODEL AND FORECAST RESULTS
# ============================================================
import joblib

joblib.dump(model, "demand_forecast_model.pkl")
print("Saved demand_forecast_model.pkl")

results = test_eval[["store_id", "product_id", "date", "units_sold", "prediction"]].copy()
results["abs_error"] = (results["units_sold"] - results["prediction"]).abs()
results.to_csv("forecast_results.csv", index=False)
print("Saved forecast_results.csv —", results.shape)

Saved demand_forecast_model.pkl
Saved forecast_results.csv — (1230210, 6)


In [14]:
# 13. NOTEBOOK SUMMARY
# ============================================================
print("NOTEBOOK 08 SUMMARY")
print(f"Store-product-date rows (dense, zero-filled): {len(dense_demand):,}")
print(f"Train: {len(train):,} rows | Test: {len(test_eval):,} rows (last 90 days, includes real Eid)")
print(f"Naive baseline MAE: {naive_mae:.4f} | LightGBM MAE: {model_mae:.4f} ({(1-model_mae/naive_mae)*100:.1f}% better)")
print(f"Naive baseline RMSE: {naive_rmse:.4f} | LightGBM RMSE: {model_rmse:.4f} ({(1-model_rmse/naive_rmse)*100:.1f}% better)")
print(f"Model's edge concentrates on spike days: +10.2% improvement pre-Eid vs +6.5% on normal days")
print(f"Top features: day_of_week, rolling_mean_28, trend_pct_per_year, promo_lift, gender")
print("Outputs: demand_forecast_model.pkl, forecast_results.csv")
print("\n✓ Notebook 08 completed successfully.")

NOTEBOOK 08 SUMMARY
Store-product-date rows (dense, zero-filled): 12,479,797
Train: 10,866,855 rows | Test: 1,230,210 rows (last 90 days, includes real Eid)
Naive baseline MAE: 0.2940 | LightGBM MAE: 0.2747 (6.5% better)
Naive baseline RMSE: 0.6211 | LightGBM RMSE: 0.4371 (29.6% better)
Model's edge concentrates on spike days: +10.2% improvement pre-Eid vs +6.5% on normal days
Top features: day_of_week, rolling_mean_28, trend_pct_per_year, promo_lift, gender
Outputs: demand_forecast_model.pkl, forecast_results.csv

✓ Notebook 08 completed successfully.
